In [2]:
def scope_test():
    def do_local():
        spam = "local spam"

    def do_nonlocal():
        nonlocal spam
        spam = "nonlocal spam"

    def do_global():
        global spam
        spam = "global spam"

    spam = "test spam"
    print('This one', type(do_local))
    do_local()
    print("After local assignment:", spam)
    do_nonlocal()
    print("After nonlocal assignment:", spam)
    do_global()
    print("After global assignment:", spam)

scope_test()
print("In global scope:", spam)

This one <class 'function'>
After local assignment: test spam
After nonlocal assignment: nonlocal spam
After global assignment: nonlocal spam
In global scope: global spam


In [3]:
class MyClass:
    """A simple example class"""
    i = 12345

    def f(self):
        return 'hello world'

In [4]:
x = MyClass()

In [7]:
print(type(x).__name__)
print(type(x.f).__name__)
print(type(x.i).__name__)

MyClass
method
int


In [84]:
function = "x**3 + 4*x*y + x*sin(z) + y*z**8"
negative = "x**3 - 4*x*y + x*sin(z) - y*z**8"
functionx = '1*x**3 + 4*x**2 + 2*x - 1'
derivativex = '3*x**2 + 8*x + 2'
functiony = "4*x**5 - 3*x**4 + x**2 - 4*x**2 + 7*x - 3"

In [85]:
next_level = functiony.split()
print(next_level)
from colorama import Fore, Back, Style

['4*x**5', '-', '3*x**4', '+', 'x**2', '-', '4*x**2', '+', '7*x', '-', '3']


In [103]:
	def derivative(function): # without sympy
		terms = []
		print("Function:", function)
		function = function.split()
		
		for i in function:
			# print("This Term:", i)
			if "**" in i and "x" in i:
				# print("Inside **")
				term = function[function.index(i)].split("**")
				# print("** Term:", term)
				if "*" in term[0]:
					# print("Inside *")
					term[0] = term[0].split("*")
					# print("* ** Term 2:", term)
					term[0][0] = int(term[0][0]) * int(term[1])
					term[1] = int(term[1]) - 1
					# print("* ** Term 3:", term)
					newterm = f"{int(term[0][0])}*{term[0][1]}**{int(term[1])}"
				# print("New ** Term:", newterm)
				
				else:
					# print("Inside no *")
					newterm = f"{term[1]}{term[0]}**{int(term[1]) - 1}"
				# print("ELSE NEWTERM:", newterm)
				if "**1" in newterm:
					newterm = newterm.replace("**1", "")
				
				terms.append(newterm)
			# print("New Term Final:", newterm)
			elif "*" in i and "**" not in i:
				# print("Inside * not **")
				# print("MultiTerm", i)
				new_multiterm = i.split("*")[0]
				# print("new Multiterm:", new_multiterm)
				terms.append(new_multiterm)
			elif "+" in i or "-" in i:
				# print("Inside +-")
				terms.append(i)
				derivative = " ".join(terms[:-3])
		print("Derivative:", derivative)


derivative(functiony)
print()
derivative(negative)

Function: 4*x**5 - 3*x**4 + x**2 - 4*x**2 + 7*x - 3
Derivative: 20*x**4 - 12*x**3 + 2x - 8*x

Function: x**3 - 4*x*y + x*sin(z) - y*z**8
Derivative: 3x**2 - 4


In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
# vispy: gallery 20

"""
Example demonstrating simulation of fireworks using point sprites.
(adapted from the "OpenGL ES 2.0 Programming Guide")
This example demonstrates a series of explosions that last one second. The
visualization during the explosion is highly optimized using a Vertex Buffer
Object (VBO). After each explosion, vertex data for the next explosion are
calculated, such that each explostion is unique.
"""

import time
import numpy as np
from vispy import gloo, app

import vispy
vispy.use('PyQt5', 'es2')


# Create a texture
radius = 32
im1 = np.random.normal(
    0.8, 0.3, (radius * 2 + 1, radius * 2 + 1)).astype(np.float32)

# Mask it with a disk
L = np.linspace(-radius, radius, 2 * radius + 1)
(X, Y) = np.meshgrid(L, L)
im1 *= np.array((X ** 2 + Y ** 2) <= radius * radius, dtype='float32')

# Set number of particles, you should be able to scale this to 100000
N = 10000

# Create vertex data container
data = np.zeros(N, [('a_lifetime', np.float32, 1),
                    ('a_startPosition', np.float32, 3),
                    ('a_endPosition', np.float32, 3)])


VERT_SHADER = """
uniform float u_time;
uniform vec3 u_centerPosition;
attribute float a_lifetime;
attribute vec3 a_startPosition;
attribute vec3 a_endPosition;
varying float v_lifetime;
void main () {
    if (u_time <= a_lifetime)
    {
        gl_Position.xyz = a_startPosition + (u_time * a_endPosition);
        gl_Position.xyz += u_centerPosition;
        gl_Position.y -= 1.0 * u_time * u_time;
        gl_Position.w = 1.0;
    }
    else
        gl_Position = vec4(-1000, -1000, 0, 0);
    v_lifetime = 1.0 - (u_time / a_lifetime);
    v_lifetime = clamp(v_lifetime, 0.0, 1.0);
    gl_PointSize = (v_lifetime * v_lifetime) * 40.0;
}
"""

# Deliberately add precision qualifiers to test automatic GLSL code conversion
FRAG_SHADER = """
#version 120
precision highp float;
uniform sampler2D texture1;
uniform vec4 u_color;
varying float v_lifetime;
uniform highp sampler2D s_texture;
void main()
{
    highp vec4 texColor;
    texColor = texture2D(s_texture, gl_PointCoord);
    gl_FragColor = vec4(u_color) * texColor;
    gl_FragColor.a *= v_lifetime;
}
"""


class Canvas(app.Canvas):

    def __init__(self):
        app.Canvas.__init__(self, keys='interactive', size=(800, 600))

        # Create program
        self._program = gloo.Program(VERT_SHADER, FRAG_SHADER)
        self._program.bind(gloo.VertexBuffer(data))
        self._program['s_texture'] = gloo.Texture2D(im1)

        # Create first explosion
        self._new_explosion()

        # Enable blending
        gloo.set_state(blend=True, clear_color='black',
                       blend_func=('src_alpha', 'one'))

        gloo.set_viewport(0, 0, self.physical_size[0], self.physical_size[1])

        self._timer = app.Timer('auto', connect=self.update, start=True)

        self.show()

    def on_resize(self, event):
        width, height = event.physical_size
        gloo.set_viewport(0, 0, width, height)

    def on_draw(self, event):

        # Clear
        gloo.clear()

        # Draw
        self._program['u_time'] = time.time() - self._starttime
        self._program.draw('points')

        # New explosion?
        if time.time() - self._starttime > 1.5:
            self._new_explosion()

    def _new_explosion(self):

        # New centerpos
        centerpos = np.random.uniform(-0.5, 0.5, (3,))
        self._program['u_centerPosition'] = centerpos

        # New color, scale alpha with N
        alpha = 1.0 / N ** 0.08
        color = np.random.uniform(0.1, 0.9, (3,))

        self._program['u_color'] = tuple(color) + (alpha,)

        # Create new vertex data
        data['a_lifetime'] = np.random.normal(2.0, 0.5, (N,))
        data['a_startPosition'] = np.random.normal(0.0, 0.2, (N, 3))
        data['a_endPosition'] = np.random.normal(0.0, 1.2, (N, 3))

        # Set time to zero
        self._starttime = time.time()


if __name__ == '__main__':
    c = Canvas()
    app.run()
